# ESR under unseen conditions — Kaggle runner

All code lives in the GitHub repo; this notebook only clones it and runs the scripts.

**Before running:** *Add Input* → `yousirui1/fsd50k`; *Settings* → **Accelerator: GPU T4**, **Internet: On**.
Run one experiment per committed version (*Save Version → Save & Run All*): `cnn_baseline`, `effnet_standard`, `effnet_robust`.

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/<your-user>/<your-repo>.git"  # <- your repo
BRANCH = "main"
!git clone -b $BRANCH $REPO_URL /kaggle/working/esr
%cd /kaggle/working/esr
!git pull origin $BRANCH
!pip install -q "timm>=1.0"

## 2. Settings

In [ ]:
EXPERIMENT = "effnet_robust"   # cnn_baseline | effnet_standard | effnet_robust
DATA_PATH = "/kaggle/input/datasets/yousirui1/fsd50k/fsd50k"  # or omit --data-path to auto-discover
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Smoke test (~15 min): 2,000 train clips, 1,000 eval clips, 2 epochs → `/kaggle/working/outputs_quick`

In [ ]:
!python scripts/train.py --experiment $EXPERIMENT --data-path $DATA_PATH --quick
!python scripts/evaluate.py --experiment $EXPERIMENT --data-path $DATA_PATH --quick --severities 2

## 4. Full run → `/kaggle/working/outputs/<experiment>/`
Resumes automatically from `last.pt`. After a 12 h timeout, add the previous version's output as an input and append
`--resume-from /kaggle/input/<that-output>/outputs/<experiment>/last.pt`.

In [ ]:
!python scripts/train.py --experiment $EXPERIMENT --data-path $DATA_PATH
!python scripts/evaluate.py --experiment $EXPERIMENT --data-path $DATA_PATH

## 5. Compare experiments
Add the finished versions' outputs as inputs; this picks up every `robustness.csv` (smoke-test results are excluded).

In [ ]:
!python scripts/compare.py --out /kaggle/working/summary.csv

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
table = pd.read_csv("/kaggle/working/summary.csv")
display(table.round(4))
table.set_index("experiment")[["clean", "seen", "unseen"]].plot.bar(title="mAP: clean vs seen vs unseen conditions")
plt.ylabel("mAP"); plt.show()